In [1]:
text = """The Imperial Russian Navy (Russian: Российский императорский флот) operated as the navy of the Russian Tsardom and later the Russian Empire from 1696 to 1917.[c] Formally established in 1696, it lasted until being dissolved in the wake of the February Revolution and the declaration of the Russian Republic in 1917. It developed from a smaller force that had existed prior to Tsar Peter the Great's founding of the modern Russian navy during the Second Azov campaign in 1696[3], and expanded in the second half of the 18th century before reaching its peak strength by the early part of the 19th century, behind only the British and French fleets in terms of size.

The Imperial Navy drew its officers from the aristocracy of the Empire, who belonged to the state Russian Orthodox Church. Young aristocrats began to be trained for leadership at a national naval boarding school, the Naval Cadet Corps. From 1818 on, only officers of the Imperial Russian Navy were appointed to the position of Chief Manager of the Russian-American Company, based in Russian America (present-day Alaska) for colonization and fur-trade development. Although the early Imperial Navy initially employed paid foreign sailors, the government began to recruit native-born sailors as conscripts, drafted (as were men to serve in the army). Service in the navy was lifelong before the 1874 decree on conscription limited the service term to six years at most. Many naval commanders and recruits came from Imperial Russia's non-Russian lands with maritime traditions—Finland and (especially) the Baltic governorates.[citation needed]

The Russian Navy went into a period of decline due to the Empire's slow technical and economic development in the first half of the 19th century. It had a revival in the latter part of the century during the reign of Emperor Nicholas II (r. 1894–1917), but most of its Pacific Fleet (along with the Baltic Fleet sent to the Far East) was destroyed in the disastrous Russo-Japanese War of 1904–1905.[4] Nicholas II, who was a naval enthusiast, had a major role in both the build up of the navy before the war with Japan and the rebuilding of it in the decade after.[5]

The navy had mixed experiences during the First World War, with the Germans generally gaining the upper hand in the Baltic Sea, while the Russians took control of the Black Sea. The Russian Baltic Fleet mostly stayed on the defensive, but the Black Sea Fleet's attacks on Ottoman merchant shipping nearly cut off the coal supply to Constantinople and threatened the Ottoman Empire's ability to stay in the war.[6][7] The Russian Revolution marked the end of the Imperial Navy; the Russian Provisional Government carried out reforms to the navy and its command structure, including the removal of imperial references from its rank insignia. Its officers had mostly aligned with the emperor, and the sailors split to fight on either side during the Russian Civil War of 1917–1922. The Soviet Navy, established as the Red Fleet in 1918 after the Revolution, took over the available surviving ships that did not evacuate from Crimea."""

In [29]:
vocab_size = 1024
num_merges = vocab_size - 256
num_merges

768

### Text chunking

In [30]:
GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

# GPT-4 style pre-tokenization regex: refines GPT-2's pattern (case-insensitive contractions,
# caps numbers to 1-3 digits, handles newlines specially, uses possessive quantifiers for speed).
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
GPT4_SPECIAL_TOKENS = {
    '<|endoftext|>': 100257,
    '<|fim_prefix|>': 100258,
    '<|fim_middle|>': 100259,
    '<|fim_suffix|>': 100260,
    '<|endofprompt|>': 100276
}

In [31]:
import regex as re
compile_pattern = re.compile(GPT4_SPLIT_PATTERN)
text_chunks = re.findall(compile_pattern, text)
print(text_chunks)

['The', ' Imperial', ' Russian', ' Navy', ' (', 'Russian', ':', ' Российский', ' императорский', ' флот', ')', ' operated', ' as', ' the', ' navy', ' of', ' the', ' Russian', ' Tsardom', ' and', ' later', ' the', ' Russian', ' Empire', ' from', ' ', '169', '6', ' to', ' ', '191', '7', '.[', 'c', ']', ' Formally', ' established', ' in', ' ', '169', '6', ',', ' it', ' lasted', ' until', ' being', ' dissolved', ' in', ' the', ' wake', ' of', ' the', ' February', ' Revolution', ' and', ' the', ' declaration', ' of', ' the', ' Russian', ' Republic', ' in', ' ', '191', '7', '.', ' It', ' developed', ' from', ' a', ' smaller', ' force', ' that', ' had', ' existed', ' prior', ' to', ' Tsar', ' Peter', ' the', ' Great', "'s", ' founding', ' of', ' the', ' modern', ' Russian', ' navy', ' during', ' the', ' Second', ' Azov', ' campaign', ' in', ' ', '169', '6', '[', '3', '],', ' and', ' expanded', ' in', ' the', ' second', ' half', ' of', ' the', ' ', '18', 'th', ' century', ' before', ' reaching

### Raw text encoding

In [32]:
ids = [list(ch.encode('utf-8')) for ch in text_chunks]
print(ids)

[[84, 104, 101], [32, 73, 109, 112, 101, 114, 105, 97, 108], [32, 82, 117, 115, 115, 105, 97, 110], [32, 78, 97, 118, 121], [32, 40], [82, 117, 115, 115, 105, 97, 110], [58], [32, 208, 160, 208, 190, 209, 129, 209, 129, 208, 184, 208, 185, 209, 129, 208, 186, 208, 184, 208, 185], [32, 208, 184, 208, 188, 208, 191, 208, 181, 209, 128, 208, 176, 209, 130, 208, 190, 209, 128, 209, 129, 208, 186, 208, 184, 208, 185], [32, 209, 132, 208, 187, 208, 190, 209, 130], [41], [32, 111, 112, 101, 114, 97, 116, 101, 100], [32, 97, 115], [32, 116, 104, 101], [32, 110, 97, 118, 121], [32, 111, 102], [32, 116, 104, 101], [32, 82, 117, 115, 115, 105, 97, 110], [32, 84, 115, 97, 114, 100, 111, 109], [32, 97, 110, 100], [32, 108, 97, 116, 101, 114], [32, 116, 104, 101], [32, 82, 117, 115, 115, 105, 97, 110], [32, 69, 109, 112, 105, 114, 101], [32, 102, 114, 111, 109], [32], [49, 54, 57], [54], [32, 116, 111], [32], [49, 57, 49], [55], [46, 91], [99], [93], [32, 70, 111, 114, 109, 97, 108, 108, 121], [32, 

In [33]:
# # text = "hello"
# ids = list(text.encode("utf-8"))
# print(ids)

### Vocab Init

    This created Unicode 👇

In [34]:
vocab = {}

for i in range(256):
    char = chr(i)
    vocab[i] = char.encode('utf-8')

print(vocab)


{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

     This created Ascii 👇

In [35]:
# vocab = {}

# for i in range(256):
#     vocab[i] = bytes([i])

# print(vocab)

In [36]:
merges = {}
vocab = {i: bytes([i]) for i in range(256)}
vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [37]:
def get_stats(ids, count=None):
    count = {} if count is None else count
    for pair in zip(ids, ids[1:]):
        count[pair] = count.get(pair, 0) + 1
    return count

In [38]:
stats = {}
for i in ids:
    get_stats(i, stats)

stats

{(84, 104): 7,
 (104, 101): 75,
 (32, 73): 11,
 (73, 109): 6,
 (109, 112): 16,
 (112, 101): 16,
 (101, 114): 44,
 (114, 105): 24,
 (105, 97): 29,
 (97, 108): 30,
 (32, 82): 22,
 (82, 117): 19,
 (117, 115): 21,
 (115, 115): 20,
 (115, 105): 26,
 (97, 110): 49,
 (32, 78): 10,
 (78, 97): 8,
 (97, 118): 18,
 (118, 121): 13,
 (32, 40): 6,
 (32, 208): 2,
 (208, 160): 1,
 (160, 208): 1,
 (208, 190): 3,
 (190, 209): 3,
 (209, 129): 4,
 (129, 209): 1,
 (129, 208): 3,
 (208, 184): 4,
 (184, 208): 4,
 (208, 185): 3,
 (185, 209): 1,
 (208, 186): 2,
 (186, 208): 2,
 (208, 188): 1,
 (188, 208): 1,
 (208, 191): 1,
 (191, 208): 1,
 (208, 181): 1,
 (181, 209): 1,
 (209, 128): 2,
 (128, 208): 1,
 (208, 176): 1,
 (176, 209): 1,
 (209, 130): 2,
 (130, 208): 1,
 (128, 209): 1,
 (32, 209): 1,
 (209, 132): 1,
 (132, 208): 1,
 (208, 187): 1,
 (187, 208): 1,
 (32, 111): 37,
 (111, 112): 5,
 (114, 97): 11,
 (97, 116): 19,
 (116, 101): 18,
 (101, 100): 25,
 (32, 97): 33,
 (97, 115): 15,
 (32, 116): 89,
 (116, 10

In [39]:
max_pair = max(stats, key=stats.get)
print(max_pair)

(32, 116)


In [40]:
def merge(ids, max_pair, idx):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == max_pair[0] and ids[i+1] == max_pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

In [41]:
for i in range(num_merges):
    stats  = {}
    for j in ids:
        get_stats(j, stats)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    ids = [merge(chunk_ids, pair, idx) for chunk_ids in ids]
    merges[pair] = idx
    vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
    print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]}) had {stats[pair]} occurrences")


merge 1/768: (32, 116) -> 256 (b' t') had 89 occurrences
merge 2/768: (104, 101) -> 257 (b'he') had 75 occurrences
merge 3/768: (256, 257) -> 258 (b' the') had 65 occurrences
merge 4/768: (97, 110) -> 259 (b'an') had 49 occurrences
merge 5/768: (101, 114) -> 260 (b'er') had 43 occurrences
merge 6/768: (105, 110) -> 261 (b'in') had 41 occurrences
merge 7/768: (32, 111) -> 262 (b' o') had 37 occurrences
merge 8/768: (97, 108) -> 263 (b'al') had 30 occurrences
merge 9/768: (262, 102) -> 264 (b' of') had 27 occurrences
merge 10/768: (115, 105) -> 265 (b'si') had 26 occurrences
merge 11/768: (114, 101) -> 266 (b're') had 25 occurrences
merge 12/768: (101, 100) -> 267 (b'ed') had 23 occurrences
merge 13/768: (97, 114) -> 268 (b'ar') had 23 occurrences
merge 14/768: (105, 116) -> 269 (b'it') had 23 occurrences
merge 15/768: (111, 110) -> 270 (b'on') had 23 occurrences
merge 16/768: (32, 82) -> 271 (b' R') had 22 occurrences
merge 17/768: (115, 116) -> 272 (b'st') had 22 occurrences
merge 18/7

In [44]:
print(vocab)

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

In [43]:
merges

{(32, 116): 256,
 (104, 101): 257,
 (256, 257): 258,
 (97, 110): 259,
 (101, 114): 260,
 (105, 110): 261,
 (32, 111): 262,
 (97, 108): 263,
 (262, 102): 264,
 (115, 105): 265,
 (114, 101): 266,
 (101, 100): 267,
 (97, 114): 268,
 (105, 116): 269,
 (111, 110): 270,
 (32, 82): 271,
 (115, 116): 272,
 (32, 261): 273,
 (117, 115): 274,
 (97, 116): 275,
 (32, 100): 276,
 (101, 110): 277,
 (274, 265): 278,
 (97, 118): 279,
 (259, 100): 280,
 (278, 259): 281,
 (32, 119): 282,
 (105, 99): 283,
 (109, 112): 284,
 (256, 111): 285,
 (111, 114): 286,
 (32, 98): 287,
 (271, 281): 288,
 (32, 110): 289,
 (32, 102): 290,
 (32, 115): 291,
 (32, 99): 292,
 (279, 121): 293,
 (32, 97): 294,
 (111, 109): 295,
 (261, 103): 296,
 (32, 280): 297,
 (108, 121): 298,
 (32, 101): 299,
 (277, 116): 300,
 (32, 73): 301,
 (32, 70): 302,
 (32, 109): 303,
 (117, 114): 304,
 (108, 101): 305,
 (32, 78): 306,
 (49, 57): 307,
 (105, 270): 308,
 (97, 100): 309,
 (284, 260): 310,
 (105, 263): 311,
 (105, 108): 312,
 (287, 1

In [ ]:
encode = {vocab[i] for i in ids}